# 8.3 Overfitting

本 notebook 使用合成表格分類資料，先建立容易過擬合的 baseline DNN，再加入 Dropout、L2 regularization 與 EarlyStopping，觀察 validation 表現是否改善。


## 1. 載入套件


In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)


## 2. 建立容易觀察過擬合的資料

資料包含少數有效特徵與較多雜訊特徵。訓練資料量刻意不大，讓模型容量過大時更容易記住訓練資料。


In [ ]:
X, y = make_classification(
    n_samples=900,
    n_features=30,
    n_informative=6,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    class_sep=1.0,
    flip_y=0.08,
    random_state=SEED,
)
X = X.astype('float32')
y = y.astype('float32')

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.35, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print(x_train.shape, x_valid.shape, x_test.shape)
print(pd.Series(y_train).value_counts().sort_index())


## 3. 建立 Baseline 模型

Baseline 模型容量偏大，且不加 regularization，用來觀察 train 與 validation curve 是否分離。


In [ ]:
def build_baseline_model():
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(x_train.shape[1],)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

baseline_model = build_baseline_model()
baseline_model.summary()


## 4. 訓練 Baseline


In [ ]:
baseline_history = baseline_model.fit(
    x_train,
    y_train,
    validation_data=(x_valid, y_valid),
    epochs=80,
    batch_size=32,
    verbose=0,
)

baseline_eval = baseline_model.evaluate(x_test, y_test, verbose=0)
print(dict(zip(baseline_model.metrics_names, baseline_eval)))


## 5. 觀察 Learning Curve


In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='valid')
    plt.title(f'{title} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='train')
    plt.plot(history.history['val_accuracy'], label='valid')
    plt.title(f'{title} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_history(baseline_history, 'Baseline')


## 6. 加入 Dropout、L2 Regularization 與 EarlyStopping


In [ ]:
def build_regularized_model():
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    regularizer = tf.keras.regularizers.l2(1e-3)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(x_train.shape[1],)),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizer),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizer),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

regularized_model = build_regularized_model()
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
    )
]

regularized_history = regularized_model.fit(
    x_train,
    y_train,
    validation_data=(x_valid, y_valid),
    epochs=80,
    batch_size=32,
    callbacks=callbacks,
    verbose=0,
)

regularized_eval = regularized_model.evaluate(x_test, y_test, verbose=0)
print('Epochs:', len(regularized_history.history['loss']))
print(dict(zip(regularized_model.metrics_names, regularized_eval)))


## 7. 比較兩個模型


In [ ]:
plot_history(regularized_history, 'Regularized')

comparison = pd.DataFrame([
    {
        'model': 'baseline',
        'epochs': len(baseline_history.history['loss']),
        'train_accuracy': baseline_history.history['accuracy'][-1],
        'best_val_accuracy': max(baseline_history.history['val_accuracy']),
        'test_accuracy': baseline_eval[1],
    },
    {
        'model': 'regularized',
        'epochs': len(regularized_history.history['loss']),
        'train_accuracy': regularized_history.history['accuracy'][-1],
        'best_val_accuracy': max(regularized_history.history['val_accuracy']),
        'test_accuracy': regularized_eval[1],
    },
])
comparison


In [ ]:
y_prob = regularized_model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test.astype(int), y_pred, digits=3))


## 8. 套用自己的資料

先訓練 baseline 並畫 learning curve，不要一開始就加入所有 regularization。當 train 與 validation 明顯分離時，再逐步加入 Dropout、L2、EarlyStopping 或資料增強。最後用 test set 做一次最終確認。
